# 5.7 — ChaseTag 1v1: the steered myoSuperMimic teacher

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/dev/tutorials/5.7_ChaseTag_Steered_Teacher.ipynb)

Play the scored game `myoChallengeChaseTagFBVs-v0` with the **steered myoSuperMimic teacher** of
`training/chasetag_teacher` on both players.

The teacher has three layers:

1. A **myoSuperMimic tracker** (`training/supermimic`) tracks a reference motion with the 354 muscles of myo_sim's
   `myofullbody`, the body of both FBVs players. It was trained on `myoMimic-v0`'s default clips: MuscleMimic's 972
   KIT training motions, retargeted onto `myofullbody`.
2. A **planner** picks a travel heading and speed: toward the opponent as chaser, away from it as runner, and turned off
   the fence.
3. A **steered reference** turns that request into the tracker's goal. It plays the KIT locomotion clips of the same
   training list under a yaw rotation, jumps between clips by motion matching, and bends the yaw a little.

This is a **playback** tutorial: the teacher reads simulator state and motion clips, so it cannot be submitted.
For a submittable 546 → 354 player:

| Step | Where |
|---|---|
| A directional walker distilled from this teacher | [5.4](./5.4_Directional_Locomotion.ipynb) |
| Export 546 → 354 and submit; the DAgger student of this teacher as a stronger start | [5.6](./5.6_ChaseTag_MyoChallenge.ipynb) |
| DAgger, the yaw-canonicalised student and the evaluation matrix | `training/chasetag_teacher` (README) |

**Data.** The steering library needs the KIT clips retargeted onto `myofullbody`. Use one of:

- a local folder in `$MYOSUITE_KIT_ROOT`, or the local retarget cache (`scripts/mimic/retarget_amass.py` fills it from
  your AMASS download);
- a Hugging Face dataset of retargeted clips: put its id in `MIMIC_DATASET` in the setup cell.

The tracker checkpoint is downloaded from `myohub/myosuite-3-baselines`. Set `$MYOSUITE_BASELINES_DIR` to a local folder laid out like that repo to use local files instead (`download_baseline_file`).

> **Citation and data:** SuperMimic (Liao et al., [arXiv:2610.28480](https://arxiv.org/abs/2610.28480)) through
> MyoSuite's port `training/supermimic`; MuscleMimic's KIT split (Li et al., 2026,
> [arXiv:2603.25544](https://arxiv.org/abs/2603.25544)). The motions come from [AMASS](https://amass.is.tue.mpg.de)
> (Mahmood et al., ICCV 2019) and the KIT whole-body human motion database (Mandery et al., 2015, 2016). AMASS-derived
> clips are for non-commercial research under the AMASS license.

## Setup

In [1]:
import os
import subprocess
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning)  # gymnasium's multi-agent checker, mujoco-warp notes

# Hugging Face dataset of KIT clips retargeted onto myofullbody (laid out as <model>/<version>/KIT/...).
# Leave empty to use $MYOSUITE_KIT_ROOT or the local retarget cache instead.
MIMIC_DATASET = os.environ.get("MYOSUITE_MIMIC_DATASET", "")  # <- fill in the dataset id
if MIMIC_DATASET:
    os.environ["MYOSUITE_MIMIC_DATASET"] = MIMIC_DATASET

QUICK_MODE = os.environ.get("MYOSUITE_FULL_1V1", "0") != "1"
STEPS = 200 if QUICK_MODE else 1000  # control steps of 0.01 s; a bout lasts at most 10 s
GIT_REF = os.environ.get("MYOSUITE_GIT_REF", "dev")
os.environ["CHASETAG_TEACHER_CACHE"] = os.environ.get(
    "CHASETAG_TEACHER_CACHE", str(Path.home() / ".cache" / "myosuite" / "chasetag_teacher")
)
if sys.platform != "darwin":
    os.environ.setdefault("MUJOCO_GL", "egl")
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

try:
    import chasetag_teacher  # noqa: F401
except ImportError:
    # MyoSuite, myoSuperMimic and the teacher project, all from the same branch.
    url = "git+https://github.com/MyoHub/myosuite.git@" + GIT_REF
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        f"myosuite[mjlab] @ {url}",
        f"myosuite-supermimic @ {url}#subdirectory=training/supermimic",
        f"chasetag-teacher @ {url}#subdirectory=training/chasetag_teacher",
        "huggingface_hub", "imageio[ffmpeg]",
    ])
print("QUICK_MODE", QUICK_MODE, "steps", STEPS)

QUICK_MODE False steps 1000


## Tracker and steering library

In [2]:
from myosuite.core.hf_io import download_baseline_file

# Verified snapshot: tracker, directional walker and ChaseTag self-play player.
BASELINES_REVISION = "7ab693c740a9dd5a812ea5c1d6dcc52e2baa3e9b"
from myosuite.envs.myo.tasks.mimic.tracking import MissingMotionData, kit_root

from chasetag_teacher.library import load_library

# Your own tracker (a training/supermimic checkpoint on myofullbody) via $CHASETAG_TEACHER_CHECKPOINT, else the published one.
TRACKER = os.environ.get("CHASETAG_TEACHER_CHECKPOINT") or download_baseline_file(
    "supermimic/tracker_myofullbody_kit_30min.pt",
    revision=BASELINES_REVISION, fallback_to_main=False,
)
print("tracker:", Path(TRACKER).name if TRACKER else None)

lib = None
try:
    KIT_ROOT = kit_root("myofullbody")  # $MYOSUITE_KIT_ROOT, the local cache, else the dataset
    lib = load_library(str(KIT_ROOT))  # built once (seconds to minutes), then cached
    print(f"steering library: {len(lib.names)} KIT locomotion clips, {lib.m.num_frames} frames")
except MissingMotionData as exc:
    print("[SKIP] no retargeted KIT clips:", exc)

tracker: tracker_myofullbody_kit_30min.pt
steering library: 860 KIT locomotion clips, 241643 frames


## Play one bout

Both players are the teacher; `agent_0` chases.

In [3]:
import numpy as np

from myosuite import make_env
from myosuite.envs.myo.tasks.challenge.chase_tag_vs.chase_tag_vs_task_config import (
    ChaseTagVsFullbodySteeredTaskConfig,
)

from chasetag_teacher.match import SmPlayer, play

frames, bout = [], None
if lib is None or TRACKER is None:
    print("[SKIP] needs the tracker and the clips (cells above)")
else:
    # The scored game; with the gait/steering shaping weights at 0 it needs no gait clip (the score is unchanged).
    config = ChaseTagVsFullbodySteeredTaskConfig(w_gait_jpos=0.0, w_gait_jvel=0.0, w_steering=0.0)
    env = make_env("myoChallengeChaseTagFBVs-v0", task_config=config, render_mode="rgb_array")
    # One teacher per player: each tracks its own body.
    players = {"agent_0": SmPlayer(str(TRACKER), lib), "agent_1": SmPlayer(str(TRACKER), lib)}

    def record(e) -> None:
        frame = e.render()
        if frame is not None:
            frames.append(frame)

    # agent_0 chases; actions get N(0, 0.02^2) noise like the evaluation matrix
    bout = play(env, players, seed=0, task="CHASE", max_steps=STEPS, record=record)
    print(f"agent_0 {bout.role0}: {bout.end} after {bout.t:.2f} s, score {bout.score0:.3f}, falls {bout.fell}")
    env.close()

agent_0 CHASE: time_up after 10.00 s, score 0.000, falls {'agent_0': None, 'agent_1': None}


## Video

In [4]:
import imageio
from IPython.display import Video, display

out = Path("chasetag_teacher.mp4")
if frames:
    with imageio.get_writer(str(out), fps=33, quality=8, macro_block_size=1) as w:
        for fr in frames[::3]:  # 100 Hz control, every third frame: real time
            w.append_data(fr)
    display(Video(str(out), embed=True))
else:
    print("no frames")

## Notes

- `chasetag_teacher.match.play` runs one bout; `SmPlayer`, `OnnxPlayer` and `IdlePlayer` are its players, so you can put
  the teacher against an ONNX file (for example your 5.6 submission) or a player that does nothing.
- In the evaluation matrix of `training/chasetag_teacher` (48 episodes per match and side), this teacher tagged a
  player that does nothing in 92 % of the bouts and won 71 % of its bouts against the shipped FBVs baseline.
- The tracker acts every 0.02 s and holds its excitation for two control steps of the game.